# datax API demo

This is a guided tour of the notebook-facing `datax` API. We will calculate one small result in Python, R, and JavaScript, move values between the three namespaces, and inspect the structured execution results.

The examples are deliberately small so that the important part is visible: `datax` gives each language a proxy for execution and shared state. Run the cells from top to bottom in a DataX notebook.

## What you will see

1. The injected `datax` object and its language proxies.
2. Synchronous execution with `datax.<language>.run(...)`.
3. Cross-language reads and calls from R and JavaScript cells.
4. Error results that can be inspected without crashing the notebook.

> **Tip:** This notebook demonstrates the live API; it is not a contract test. For exhaustive assertions, see `notebooks/test/test_datax_api.ipynb`.

## 1. Discover the injected API

The `datax` object is injected into the Python, R, and JavaScript namespaces. Each language proxy supports execution and variable access.

In [1]:
print('datax:', datax)
print('type:', type(datax).__name__)

for language in ('python', 'r', 'javascript'):
    proxy = getattr(datax, language)
    print(f'{language:>10}: run={callable(getattr(proxy, "run", None))}, '
          f'execute={callable(getattr(proxy, "execute", None))}, '
          f'vars={hasattr(proxy, "vars")}')

datax: <DataxNamespace: python=0 vars, r=0 vars, javascript=0 vars>
type: DataxNamespace
    python: run=True, execute=True, vars=True
         r: run=True, execute=True, vars=True
javascript: run=True, execute=True, vars=True


## 2. Run all three languages from Python

A call to `run(...)` returns a result envelope. The exact details can vary by backend, but `status`, `stdout`, `stderr`, and `error` are the useful fields for a demo.

In [2]:
def show_result(label, result):
    """Print the useful part of a datax execution result."""
    status = result.get('status')
    print(f'\n--- {label} ---')
    print('status:', status)
    if result.get('stdout'):
        print('stdout:', result['stdout'].strip())
    if result.get('stderr'):
        print('stderr:', result['stderr'].strip())
    if result.get('error'):
        print('error:', result['error'])
    return result

python_result = show_result('Python', datax.python.run("""
numbers = [2, 4, 6, 8]
demo_total = sum(numbers)
print('Python total:', demo_total)
"""))

r_result = show_result('R', datax.r.run("""
r_values <- c(3, 5, 8, 13)
r_mean <- mean(r_values)
cat('R mean:', r_mean, '\n')
"""))

js_result = show_result('JavaScript', datax.javascript.run("""
const jsValues = [10, 20, 30];
globalThis.jsTotal = jsValues.reduce((total, value) => total + value, 0);
console.log('JavaScript total:', jsTotal);
"""))


--- Python ---
status: success
stdout: Python total: 20

--- R ---
status: success
stdout: R mean: 7.25
JavaScript total: 60

--- JavaScript ---
status: success
stdout: JavaScript total: 60


## 3. Read shared values from Python

Language proxies are also dict-like. Use `proxy['name']` when the variable name is dynamic or when you want to make the namespace explicit.

In [3]:
print("Python namespace:", datax.python["demo_total"])
print("R namespace:", datax.r["r_mean"])
print("JavaScript namespace:", datax.javascript["jsTotal"])

combined_total = datax.python["demo_total"] + datax.javascript["jsTotal"]
print("Combined Python + JavaScript total:", combined_total)

Python namespace: 20
R namespace: 7.25
JavaScript namespace: 60
Combined Python + JavaScript total: 80


## 4. Call Python from an R cell

The same proxies are available inside an R cell through `$`. This cell reads values from the other namespaces and asks Python to create a value.

In [4]:
%%R
cat('Python total seen by R:', datax$python$demo_total, '\n')
cat('JavaScript total seen by R:', datax$javascript$jsTotal, '\n')

result <- datax$python$run("py_from_r <- 'created by an R cell'; print(py_from_r)")
cat('Python call status:', result$status, '\n')
cat('Python call output:', result$stdout, '\n')
r_message <- paste('R combined total:', datax$python$demo_total + datax$javascript$jsTotal)
cat(r_message, '\n')

Python total seen by R: 20 
JavaScript total seen by R: 60 
Python call status: success 
Python call output: created by an R cell
 
R combined total: 80 

## 5. Call R asynchronously from a JavaScript cell

JavaScript uses normal property access. Cross-language calls can be awaited with `run_async()`, which is useful when the target execution may take longer.

In [5]:
%%js
console.log('Python total seen by JS:', datax.python.demo_total);
console.log('R message seen by JS:', datax.r.r_message);

const result = await datax.r.run_async("js_requested_r_value <- 7 * 6; cat('R value for JS:', js_requested_r_value, '\\n')");
console.log('R call status:', result.status);
console.log('R call output:', result.stdout ?? result.outputs ?? '');

globalThis.jsSummary = {
  pythonTotal: datax.python.demo_total,
  rMean: datax.r.r_mean,
  requestedRValue: datax.r.js_requested_r_value
};
console.log('JS summary:', JSON.stringify(jsSummary));

Python total seen by JS: 20
R message seen by JS: R combined total: 80


Execution finished.

R value for JS: 42 R call status: completed
R call output: [
  {
    "name": "stdout",
    "output_type": "stream",
    "text": "R value for JS: 42 "
  }
]
JS summary: {"pythonTotal":20,"rMean":7.25,"requestedRValue":42}


## 6. Errors are returned as data

For exploratory work, a failed `run()` returns an inspectable result envelope. The notebook can decide what to do next instead of hiding the backend error.

In [6]:
failed_result = show_result(
    'Expected Python failure',
    datax.python.run("raise ValueError('demo error: inspect the result envelope')")
)

print('Failure captured:', failed_result.get('status') not in (0, 'success'))


--- Expected Python failure ---
status: error
Failure captured: True


## 7. Verify the final shared state

At this point, each language has contributed to the same small workflow.

In [7]:
print('Python:', {
    'demo_total': datax.python['demo_total'],
    'py_from_r': datax.python['py_from_r'],
})
print('R:', {
    'r_mean': datax.r['r_mean'],
    'r_message': datax.r['r_message'],
    'js_requested_r_value': datax.r['js_requested_r_value'],
})
print('JavaScript:', datax.javascript['jsSummary'])
print('\nDemo complete.')

Python: {'demo_total': 20, 'py_from_r': 'created by an R cell'}
R: {'r_mean': 7.25, 'r_message': 'R combined total: 80', 'js_requested_r_value': 42.0}
JavaScript: {'pythonTotal': 20, 'rMean': 7.25, 'requestedRValue': 42}

Demo complete.


### API summary

- `datax.python`, `datax.r`, and `datax.javascript` are language proxies.
- `proxy.run(code)` executes code and returns a result envelope.
- `proxy.run_async(code)` supports awaited cross-language calls.
- `proxy['name']` and `proxy.name` read values from a language namespace.
- `proxy.execute(code, ...)` is available when you need execution options such as a timeout.
- The module-level helpers (`datax.run_python`, `datax.run_r`, and `datax.run_javascript`) remain available for older integrations.